In [1]:
import pandas as pd
import numpy as np
from openpyxl import Workbook
from datetime import datetime
from datetime import timedelta
from openpyxl.utils import get_column_letter
import datetime  
import pdb
import pandas as pd
import numpy as np 
# from pandas.io.json import json_normalize  
from pandas import json_normalize
import json
import psycopg2
from datetime import datetime
from dateutil.relativedelta import relativedelta
from urllib.parse import quote
from sqlalchemy.engine import create_engine
import warnings
warnings.filterwarnings("ignore")
# C:\Users\saurabh.khopkar\Saurabh pythond code save path

In [2]:
# pip install install sqlalchemy

In [3]:
# sas_config = r"C:\Users\mononita.mitra\OneDrive - UGRO Capital\Desktop\Mononita Data 14-09-2023\Mononita\GroMicro Login Analysis\config_ods.json"
# sas_config = r"C:\Users\saurabh.khopkar\Downloads\config_ods 2.json"
sas_config = r"C:\Users\saurabh.khopkar\Downloads\config_ods_Saurabh.json"
def connection_from_sas():
    try:
        f = open(sas_config)
        data = json.load(f)

 

        database = data.get("database")
        user = data.get("user")
        password = data.get("password")
        host = data.get("host")
        port = data.get("port")

 

        connection = psycopg2.connect(database=database, user=user, password=password, host=host, port=port)
        return connection

 

    except Exception as error:
        print(error)

In [4]:
connection = connection_from_sas()

In [5]:
# df2.to_excel(r"C:\Users\saurabh.khopkar\Downloads\Data.xlsx", index=False)

In [6]:
sql_query_1 = """
select object_pri_key_1,equipment_make from collateral where equipment_make <>''
"""
# select object_pri_key_1,equipment_make,equipment_type from collateral where equipment_make <>''
# and object_pri_key_1 in ('4FIN22050203550050840','4FIN220623033842642227','4FIN220720022010710305')
# where b.datetime_login >='2024-09-01'
# left join laf_detail_status b on a.object_pri_key_1=b.laf_number
# where b.datetime_login >='2024-09-01'

In [7]:
df1 = pd.read_sql_query(sql_query_1, connection)

In [8]:
sql_query_2 = """
select object_pri_key_1,equipment_type from collateral where equipment_type <>''
"""


In [9]:
df2 = pd.read_sql_query(sql_query_2, connection)

In [10]:
sql_query_3 = """
SELECT object_pri_key_1,activitydescription
FROM tb_udhyam_details a
where activitydescription <>''
"""

In [11]:
df3 = pd.read_sql_query(sql_query_3, connection)

In [12]:
# df3=df3.replace('NULL',np.nan)
# df3=df3.replace(to_replace=" NULL",value=0)
# df3['equipment_make'] = df3['equipment_make'].fillna(0)

In [13]:
df1=df1.drop_duplicates()
df2=df2.drop_duplicates()
df3=df3.drop_duplicates()

In [14]:
# # df2.groupby(['object_pri_key_1','program_type']).agg({'activitydescription': ', '.join}).reset_index()
# df2 = df.groupby('object_pri_key_1').agg({'Sid':'first', 
#                              'Use_Case': ', '.join, 
#                              'Revenue':'first' }).reset_index()
df1 = df1.groupby('object_pri_key_1').agg({'equipment_make': '/ '.join
                             }).reset_index()

In [15]:
df2 = df2.groupby('object_pri_key_1').agg({'equipment_type': '/ '.join
                             }).reset_index()

In [16]:
df3 = df3.groupby('object_pri_key_1').agg({'activitydescription': '/ '.join
                             }).reset_index()

In [17]:
sql_query_4 = """
SELECT Laf_number,program_name,schemedesc,applicant_name,company_name,purpose_of_loan,case_assigned_to_id,line_of_business,agreementno as LAN
FROM laf_detail_status a
left join jayam_contract_details b on a.laf_number=b.laf_id
"""
# where a.datetime_login >='2024-09-01'

In [18]:
df4 = pd.read_sql_query(sql_query_4, connection)

In [19]:
df4.rename(columns={'laf_number':'object_pri_key_1'},inplace=True)

In [20]:
df5=pd.merge(df4,df1,on='object_pri_key_1',how='left')

In [21]:
df6=pd.merge(df5,df2,on='object_pri_key_1',how='left')

In [22]:
df7=pd.merge(df6,df3,on='object_pri_key_1',how='left')

In [23]:
# df7.to_excel(r"C:\Users\saurabh.khopkar\Downloads\LAF_Solar_Case.xlsx", index=False)

In [24]:
# df7.loc[df7['activitydescription'].isin(['SL-EV Secured UBL Program']),'Solar_Ind']=1
df7.loc[df7['program_name'].isin(['SL-EV Secured UBL Program']),'Solar_Ind']=1
df7.loc[df7['schemedesc'].isin(['EV Secured UBL']),'Solar_Ind']=1
df7.loc[df7['equipment_type'].isin(['EPC','EV']),'Solar_Ind']=1
# df7.loc[df7['activitydescription'].isin(['Transmission of electric energy',
#                                          'Treatment of waste water or sewer by means of physical, chemical or biological processes',
#                                          'Electric power generation using solar energy',
#                                          'Electric power generation by hydroelectric power plants']),'Solar_Ind']=1

In [25]:
# df7.loc[df7['activitydescription'].str.contains(['energy']),'Solar_Ind']=1
# df8=df7[df7['activitydescription'].str.contains("solar",na=False)]

In [26]:
mask = ((df7['activitydescription'].str.contains('Electric power generation using solar energy')) | 
        (df7['activitydescription'].str.contains('Transmission of electric energy')) |
        (df7['activitydescription'].str.contains('Electric power generation by hydroelectric power plants')) |
        (df7['activitydescription'].str.contains('Treatment of waste water or sewer by means of physical, chemical or biological processes')) |
        
        (df7['company_name'].str.contains('Energy')) |
        (df7['company_name'].str.contains('ENERGY')) |
        (df7['company_name'].str.contains('Solar')) |
        (df7['company_name'].str.contains('SOLAR')) |
        
        (df7['applicant_name'].str.contains('Energy')) |
        (df7['applicant_name'].str.contains('ENERGY')) |
        (df7['applicant_name'].str.contains('Solar')) |
        (df7['applicant_name'].str.contains('SOLAR')) |
        
        
        (df7['purpose_of_loan'].str.contains('solar')) |
        (df7['equipment_make'].str.contains('solar')) |
        (df7['equipment_type'].str.contains('solar')) |
        
        (df7['purpose_of_loan'].str.contains('SOLAR')) |
        (df7['equipment_make'].str.contains('SOLAR')) |
        (df7['equipment_type'].str.contains('SOLAR')) 
       )
df7.loc[mask, 'Solar_Ind'] = 1

In [27]:
df8=df7

In [28]:
# df8['Solar_Ind'] = df8['Solar_Ind'].fillna(-1)
df8.loc[mask, 'Solar_Ind'] = 1

In [29]:
df8= df8.drop(columns=['schemedesc','line_of_business','equipment_make', 
                       'equipment_type','activitydescription','company_name','applicant_name'])

In [30]:
df8.columns

Index(['object_pri_key_1', 'program_name', 'purpose_of_loan',
       'case_assigned_to_id', 'lan', 'Solar_Ind'],
      dtype='object')

In [31]:
# sql_query_5 = """
# Select a.agreementno, a.agreementid, c.customername, a.productdescription, a.productflag,
#    CASE
#             WHEN a.AGREEMENTNO in ('HCFHYDSEC00001001119','HCFHYDSEC00001001150','HCFHYDSEC00001001219','HCFHYDSEC00001001285','HCFCHESEC00001001244','HCFHYDSEC00001001504','HCFBANSEC00001001692','HCFHYDUSC00001001128','HCFCHEUSC00001001151','HCFBANUSC00001001197','UGBRPSS0000046859','UGBRPSS0000068131','UGHYDSS0000031715','UGJAISS0000072023') THEN 'Emerging-Market-Business' 
#             WHEN a.SCHEMEDESC in ('UNSECURED - FT CASH','DIGITAL LENDING PAISA BAZAR UNSECURED','UNSECURED- LENDINGKART','RECUR - UNSECURED','ANTWORK-UNSECURED','COLENDING RES UNSEC','TURNO - PNA MACHINERY SECURED','VIDYUTTECH SERVICES - SECURED','Rapipay','UNSECURED- CREDRIGHT','Rapipay','SECURED - IREP','UNSECURED - INDIFI','UNSECURED- CREDRIGHT','SECURED - JFL','SECURED - IREP','SECURED - JFL','TURNO - PNA MACHINERY SECURED','UNSECURED- CREDRIGHT','SECURED - JFL','DHANVARSHA - UNSECURED','Rapipay','Subhalakshmi Finance','SWARA UNSECURED','UNSECURED - INDIFI') THEN 'Partnership_Colending' 
#             WHEN a.SCHEMEDESC in ('ORIGA RENT RECEIVABLE','SECURED - RENTAL RECEIVABLE ASSIST','ECLGS - Plant & Machinery','SECURED - RENTAL RECEIVABLE','ECLGS - PLANT AND MACHINERY','MACHINERY RES MSME','MACHINERY RESTRUCTURING','NORMAL INCOME PROGRAM MACHINE','SECURED - RENTAL RECEIVABLE CONNECT RESIDUARY','EV Secured UBL','UNSECURED MACHINERY AND EQUIPMENT LOAN') THEN 'MACHINERY LOAN' 
#             WHEN a.SCHEMEDESC in ('SECURED - SCF','OYO Secured Loan','SEC Program','PROGRAM SEC','LOW TV - SCF','MSME RES UNSEC - SCF','UNSECURE COVRES2 - SCF','ECLGS - SCF') THEN 'SCF' 
#             WHEN a.PRODUCTDESCRIPTION='SUPPLY CHAIN RECEIVABLE' THEN 'SCF' 
#             WHEN a.PRODUCTDESCRIPTION IN ('MACHINERY LOAN') AND a.SCHEMEDESC not in ('TURNO - PNA MACHINERY SECURED') THEN 'MACHINERY LOAN'
#             WHEN a.SCHEMEDESC in ('MTL - MACHINERY PROGRAM','TURNOVER PROGRAM MACHINARY','LOW LTV MACHINERY LOAN') and LOWER(a.BRANCHDESC) in ('ahmedabad','bangalore','chennai','delhi','hyderabad','jaipur','kolkata','mumbai','pune','chandigarh','baroda prime','coimbatore prime','jodhpur prime','madurai prime','surat prime','udaipur prime','hubli prime','nashik','raipur','vizag','indore','mysore prime','nagpur','vijayawada','rajkot prime','agra prime','bhopal prime','lucknow prime','ludhiana','salem prime','kanpur prime') THEN 'MACHINERY LOAN'
#             WHEN a.PRODUCTDESCRIPTION in ('DIRECT ASSIGNMENT UN-SECURED','DIRECT ASSIGNMENT SECURED','AVANSE') OR a.SCHEMEDESC IN ('NORMAL INCO PROG SEC - ONWARD LEND','SECURE COVRES1- ONWARD LENDING','AVANSE','AVANSE AVCOVRES','JUMBO Secured','Lending Kart Direct Assignment','UP MONEY Direct Assignment') THEN 'DA_PA' 
#             WHEN a.SCHEMEDESC in ('MACHINERY PEERLESS BUYOUT','SECURED PEERLESS BUYOUT','UNSECURED PEERLESS BUYOUT') THEN 'DA_PA'
#             WHEN a.PRODUCTDESCRIPTION in ('UN-SECURED CO-LENDING','SECURED CO-LENDING','P&A UNSECURED','P&A SECURED') THEN 'Partnership_Colending' 
#             WHEN a.SCHEMEDESC IN ('SaathiPlus - Secured', 'SAATHIPRIME - SECURED' ,'SAATHIPLUS - SECURED','SaathiPrime - Secured','Secured - Unnati', 'ECLGS - SAATHI','SAHYOG SECURED - AIP', 'SAHYOG SECURED - ABB', 'SAHYOG SECURED - ITR','SAHYOG SECURED - GST','LOW LTV MACHINERY LOAN','SAATHIPLUS - SECURED', 'SaathiPlus - Secured','AUTO TOP UP PROG - SANJEEVANI SEC', 'SECURED - SANJEEVANI') and LOWER(a.BRANCHDESC) in ('ahmedabad', 'bangalore','chennai', 'delhi','hyderabad','jaipur', 'kolkata', 'mumbai','pune', 'jodhpur prime','chandigarh','ludhiana','udaipur prime','vizag','surat prime','indore','nagpur','madurai prime','baroda prime','rajkot prime','coimbatore prime','mysore prime','hubli prime','vijayawada','raipur','nashik','lucknow prime','bhopal prime','jabalpur prime','kanpur prime','kota prime','salem prime','valsad prime','varanasi prime') THEN 'Sanjeevani_Secured'
#             WHEN a.SCHEMEDESC IN ('ABB PROGRAM BANK UNSECURED','UNSECURED - EMERGENCY CREDIT LINE', 'UNSECURE COVRES1', 'UNSECURE COVRES2','SMALL BUSINESS RES UNSEC','TURNOVER PROGRAM UNSECURED','DOCTOR PROGRAM - UNSECURED', 'NORMAL INCOME PROGRAM UNSECURE', 'TURNOVER PROGRAM GST UNSEC', 'Turnover Program P&N Unsec', 'Direct to Customer  Unsecured', 'DOCTOR PROGRAM UNSECURED', 'TURNOVER PROGRAM BANK UNSEC','UNSECURED - LOANFRAME','SBL - UNSECURED' ) and  LOWER(a.BRANCHDESC) in ('ahmedabad','bangalore','chennai','delhi','hyderabad','jaipur','kolkata','mumbai','pune','chandigarh','baroda prime','coimbatore prime','coimbatore prime','jodhpur prime','madurai prime','surat prime','udaipur prime','hubli prime','nashik','raipur','vizag','indore','mysore prime','nagpur','vijayawada','rajkot prime','salem prime','lucknow prime','bhopal prime','jabalpur prime','kanpur prime','kota prime','salem prime','valsad prime','varanasi prime','ludhiana') THEN 'Sanjeevani_UNSecured'
#             WHEN a.SCHEMEDESC IN ('SBL - UNSECURED') and  LOWER(a.BRANCHDESC) in ('ahmedabad','bangalore','chennai','delhi','hyderabad','jaipur','kolkata','mumbai','pune','chandigarh','baroda prime','coimbatore prime','coimbatore prime','jodhpur prime','madurai prime','surat prime','udaipur prime','hubli prime','nashik','raipur','vizag','indore','mysore prime','nagpur','vijayawada','rajkot prime') THEN 'SBL-Unsecured'                
#             WHEN a.SCHEMEDESC IN ('AUTO TOP UP PROG - SANJEEVANI UNSEC','DHANVARSHA - UNSECURED','Rapipay','SAATHIPLUS - SECURED','SECURED - IREP','SECURED - JFL','SWARA UNSECURED','TURNO - PNA MACHINERY SECURED','UNSECURED - INDIFI','UNSECURED - SANJEEVANI','UNSECURED- CREDRIGHT', 'EDI Program') and  LOWER(a.BRANCHDESC) in ('Yamunanagar','ahmedabad two','ajmer','barmer','baroda','belgaum','bhavnagar','bikaner','bommasandra','channagiri','channapatna','chikodi','chitradurga','chomu','coimbatore','davanagere','dharapuram','didwana','dindigul','gadwal micro','ganganagar','gokak','himmatnagar','hospet','jagatpura','jagtial','jaitram','jamnagar','jhotwara','jodhpur','junagadh','kamareddy','kanchipuram','karimnagar','kurnool','mahabubnagar','mahadevpura','miryalaguda','modasa','morbi','mudhol','mysore','nadiad','nagercoil','nalgonda','nizamabad','odhav','ottanchathiram','pali','panipat','peenya','perundurai','rajkot one','rr nagar','salem','siddipet','sikar','sivakasi','sumerpur','surat one','suryapet','tambaram','tenkasi','theni','thoothukudi','tirunelveli','tirupati','vellore','vijayapura','warangal','valsad','hoskote','ambattur','kota micro','agra micro','bhimavaram','indore micro','jabalpur micro','jhalawar micro','kovilpatti micro','lucknow micro','mira road micro','panvel 2 micro','panvel micro','rajendra nagar micro','vidisha micro','gorakhapur micro','varanasi','bhopal micro','gadwal micro','vidisha micro','proddatur') THEN 'Emerging-Market-Business'
#             WHEN a.SCHEMEDESC IN ('MSME RES UNSEC','AUTO TOP UP PROG - SANJEEVANI UNSEC','UNSECURED - SANJEEVANI','Purushottam DA Buyout') THEN 'Sanjeevani_UNSecured'
#             WHEN a.SCHEMEDESC IN ('GRO-MICRO-SECURED', 'GRO-MICRO-UNSECURED','ECLGS - GRO-MICRO-UNSECURED', 'ABB PROGRAM BANK UNSECURED','UNSECURED - EMERGENCY CREDIT LINE', 'UNSECURE COVRES1', 'UNSECURE COVRES2','SMALL BUSINESS RES UNSEC','TURNOVER PROGRAM UNSECURED', 'SBL - UNSECURED', 'SAATHIPRIME - SECURED' ,'SaathiPrime - Secured','Secured - Unnati','ECLGS - SAATHI','DOCTOR PROGRAM - UNSECURED', 'NORMAL INCOME PROGRAM UNSECURE', 'TURNOVER PROGRAM GST UNSEC', 'Turnover Program P&N Unsec', 'Direct to Customer  Unsecured', 'DOCTOR PROGRAM UNSECURED', 'TURNOVER PROGRAM BANK UNSEC','UNSECURED - LOANFRAME','SAHYOG SECURED - AIP', 'SAHYOG SECURED - ABB','SAHYOG SECURED - GST','LOW LTV MACHINERY LOAN','MICRO SECURED BLI','MICRO UNSECURED BLI','AUTO TOP UP PROG - SANJEEVANI SEC','MTL - MACHINERY PROGRAM','SECURED - SANJEEVANI','TURNOVER PROGRAM MACHINARY','Micro Secured-EM 2') THEN 'Emerging-Market-Business'
#             WHEN a.SCHEMEDESC IN ('TURNOVER PROGRAM BANK SEC', 'TURNOVER PROGRAM P&N SEC',  'TURNOVER PROGRAM GST SEC', 'EDI SECURED LOAN', 'Turnover Program Secured','LOW TV SECURED','SECURED - EMERGENCY CREDIT LINE','NORMAL INCOME PROGRAM SECURED','SECURE COVRES1','SMALL BUSINESS RES SEC','ABB PROGRAM BANK SECURED','SECURED - PRATHAM', 'MSME RES SEC' ) THEN 'Sanjeevani_Secured' 
#             ELSE 'NA' 
#             END AS NEWPRODUCT,
#         a.schemedesc, b.prepay_penalty, a.branchdesc, a.effrate, a.disbursaldate, b.future_principle, a.closuredate, d.case_assigned_to_id,u.HOME_LOCATION as home_location,
# 		 case 
# 		 	when b.remarks = '' or b.remarks is null then disb.status
# 			else b.remarks
# 		end as closure_Remarks,
# 		case when CANCELLED.status is null then 'NO' ELSE 'YES' end as cancelled
# 		from jayam_contract_details a 
# 		inner join jayam_contract_termination_dtl b on a.agreementid = b.agreementid 
# 		left join jayam_agreement_customer_info c on c.agreementid = a.agreementid and c.seq = 1 
#         left join laf_detail_status d on d.laf_number = a.laf_id 
# 		left join jayam_disbursement_schedule disb on disb.agreementid = a.agreementid and disb.status = 'Disbursement Cancelled'
#         left join (SELECT user_id, string_agg( distinct HOME_LOCATION, ',') HOME_LOCATION FROM public.los_user_mst
# 	group by user_id) u on u.USER_ID=d.case_assigned_to_id
# 	left join (
# 	SELECT DISTINCT
# 		AGREEMENTID,
# 		STATUS
# 	FROM
# 		(
# 			SELECT DISTINCT
# 				DISB.AGREEMENTID,
# 				'Disbursement Cancelled' AS STATUS
# 			FROM
# 				JAYAM_DISBURSEMENT_SCHEDULE DISB
# 				INNER JOIN JAYAM_CONTRACT_DETAILS CONT_DET ON CONT_DET.AGREEMENTID = DISB.AGREEMENTID
# 			WHERE
# 				DISB.STATUS IN ('Disbursement Cancelled')
# 				AND EXTRACT(
# 					MONTH
# 					FROM
# 						CONT_DET.CLOSUREDATE
# 				) = EXTRACT(
# 					MONTH
# 					FROM
# 						CONT_DET.DISBURSALDATE
# 				)
# 				AND EXTRACT(
# 					YEAR
# 					FROM
# 						CONT_DET.CLOSUREDATE
# 				) = EXTRACT(
# 					YEAR
# 					FROM
# 						CONT_DET.DISBURSALDATE
# 				)
# 			UNION
# 			SELECT DISTINCT
# 				A.AGREEMENTID,
# 				'OTC Loan Cancellation' AS STATUS
# 			FROM
# 				JAYAM_CONTRACT_TERMINATION_DTL B
# 				INNER JOIN JAYAM_CONTRACT_DETAILS A ON A.AGREEMENTID = B.AGREEMENTID
# 				AND (
# 					EXTRACT(
# 						MONTH
# 						FROM
# 							A.CLOSUREDATE
# 					) = EXTRACT(
# 						MONTH
# 						FROM
# 							A.DISBURSALDATE
# 					)
# 					AND EXTRACT(
# 						YEAR
# 						FROM
# 							A.CLOSUREDATE
# 					) = EXTRACT(
# 						YEAR
# 						FROM
# 							A.DISBURSALDATE
# 					)
# 					AND B.REMARKS IN ('OTC Loan Cancellation')
# 				)
# 		) TEMP
# ) AS CANCELLED on CANCELLED.agreementid = a.agreementid 
#         where a.closuredate between '2024-09-01' and '2025-09-30'

# """

# # where a.closuredate between '2024-09-01' and CURRENT_DATE
# #   d.applicant_name,
# #    d.company_name,
# #    d.purpose_of_loan,
# #    d.program_name
# #  e.user_id,

In [32]:
# import sqlite3

In [33]:
# # connection = connection_from_sas()
# conn = sqlite3.connect("connection_from_sas()")

In [34]:

sql_query_5 = """
Select a.agreementno, c.customername, a.productdescription, a.productflag, a.schemedesc, b.prepay_penalty,a.branchdesc, a.effrate, a.disbursaldate, b.future_principle, a.closuredate,((a.closuredate - a.disbursaldate )/365) AS Day_Diff,
        CASE 

   WHEN a.SCHEMEDESC in ('UNSECURED - FT CASH','DIGITAL LENDING PAISA BAZAR UNSECURED','UNSECURED- LENDINGKART','RECUR - UNSECURED','ANTWORK-UNSECURED','COLENDING RES UNSEC','TURNO - PNA MACHINERY SECURED','VIDYUTTECH SERVICES - SECURED') THEN 'Partnership_Colending'

   WHEN a.SCHEMEDESC in ('ORIGA RENT RECEIVABLE','SECURED - RENTAL RECEIVABLE ASSIST','ECLGS - Plant & Machinery','SECURED - RENTAL RECEIVABLE','ECLGS - PLANT AND MACHINERY','TURNOVER PROGRAM MACHINARY','MTL - MACHINERY PROGRAM','MACHINERY RES MSME','MACHINERY RESTRUCTURING','NORMAL INCOME PROGRAM MACHINE','SECURED - RENTAL RECEIVABLE CONNECT RESIDUARY') THEN 'MACHINERY LOAN'

   WHEN a.SCHEMEDESC in ('SECURED - SCF','OYO Secured Loan','SEC Program','PROGRAM SEC','LOW TV - SCF','MSME RES UNSEC - SCF','UNSECURE COVRES2 - SCF','ECLGS - SCF') THEN 'SCF'

   WHEN a.PRODUCTDESCRIPTION='SUPPLY CHAIN RECEIVABLE' THEN 'SCF'

   WHEN a.PRODUCTDESCRIPTION IN ('MACHINERY LOAN','MACHINERY') AND a.SCHEMEDESC not in ('TURNO - PNA MACHINERY SECURED') THEN 'MACHINERY LOAN'

   WHEN a.PRODUCTDESCRIPTION in ('DIRECT ASSIGNMENT UN-SECURED','DIRECT ASSIGNMENT SECURED','AVANSE') OR a.SCHEMEDESC IN ('NORMAL INCO PROG SEC - ONWARD LEND','SECURE COVRES1- ONWARD LENDING','AVANSE','AVANSE AVCOVRES','JUMBO Secured','Lending Kart Direct Assignment','UP MONEY Direct Assignment') THEN 'DA_PA'

   WHEN a.PRODUCTDESCRIPTION in ('UN-SECURED CO-LENDING','SECURED CO-LENDING','P&A UNSECURED','P&A SECURED') THEN 'Partnership_Colending'

   WHEN a.SCHEMEDESC IN ('SaathiPlus - Secured', 'SAATHIPRIME - SECURED' ,'SAATHIPLUS - SECURED','SaathiPrime - Secured','Secured - Unnati', 'ECLGS - SAATHI','SAHYOG SECURED - AIP', 'SAHYOG SECURED - ABB', 'SAHYOG SECURED - ITR','SAHYOG SECURED - GST','LOW LTV MACHINERY LOAN') and LOWER(a.BRANCHDESC) in ('ahmedabad', 'bangalore','chennai', 'delhi','hyderabad','jaipur', 'kolkata', 'mumbai','pune', 'jodhpur prime','chandigarh','ludhiana','udaipur prime','vizag','surat prime','indore','nagpur','madurai prime','baroda prime','rajkot prime','coimbatore prime','mysore prime','hubli prime','vijayawada','raipur') THEN 'Saathi_Secured'

   WHEN a.SCHEMEDESC IN ('ABB PROGRAM BANK UNSECURED','UNSECURED - EMERGENCY CREDIT LINE', 'UNSECURE COVRES1', 'UNSECURE COVRES2','SMALL BUSINESS RES UNSEC','TURNOVER PROGRAM UNSECURED', 'SBL - UNSECURED','DOCTOR PROGRAM - UNSECURED', 'UNSECURED - SANJEEVANI', 'NORMAL INCOME PROGRAM UNSECURE', 'TURNOVER PROGRAM GST UNSEC', 'Turnover Program P&N Unsec', 'Direct to Customer  Unsecured', 'DOCTOR PROGRAM UNSECURED', 'TURNOVER PROGRAM BANK UNSEC','UNSECURED - LOANFRAME' ) and  LOWER(a.BRANCHDESC) in ('ahmedabad','bangalore','chennai','delhi','hyderabad','jaipur','kolkata','mumbai','pune','chandigarh','baroda prime','coimbatore prime','coimbatore prime','jodhpur prime','madurai prime','surat prime','udaipur prime','hubli prime','nashik','raipur','vizag','indore','mysore prime','nagpur','vijayawada','rajkot prime') THEN 'Sanjeevani_UNSecured'

   WHEN a.SCHEMEDESC IN ('UNSECURED - SANJEEVANI','MSME RES UNSEC','UNSECURED PEERLESS BUYOUT') THEN 'Sanjeevani_UNSecured'

   WHEN a.SCHEMEDESC IN ('GRO-MICRO-SECURED', 'GRO-MICRO-UNSECURED','ECLGS - GRO-MICRO-UNSECURED', 'ABB PROGRAM BANK UNSECURED','UNSECURED - EMERGENCY CREDIT LINE', 'UNSECURE COVRES1', 'UNSECURE COVRES2','SMALL BUSINESS RES UNSEC','TURNOVER PROGRAM UNSECURED', 'SBL - UNSECURED', 'SaathiPlus - Secured', 'SAATHIPRIME - SECURED' ,'SAATHIPLUS - SECURED','SaathiPrime - Secured','Secured - Unnati','ECLGS - SAATHI','DOCTOR PROGRAM - UNSECURED', 'UNSECURED - SANJEEVANI', 'NORMAL INCOME PROGRAM UNSECURE', 'TURNOVER PROGRAM GST UNSEC', 'Turnover Program P&N Unsec', 'Direct to Customer  Unsecured', 'DOCTOR PROGRAM UNSECURED', 'TURNOVER PROGRAM BANK UNSEC','UNSECURED - LOANFRAME','SAHYOG SECURED - AIP', 'SAHYOG SECURED - ABB','SAHYOG SECURED - GST','LOW LTV MACHINERY LOAN','MICRO SECURED BLI','MICRO UNSECURED BLI') THEN 'Gro-Micro'

   WHEN a.SCHEMEDESC IN ('SECURED - SANJEEVANI', 'TURNOVER PROGRAM BANK SEC', 'TURNOVER PROGRAM P&N SEC',  'TURNOVER PROGRAM GST SEC', 'EDI SECURED LOAN', 'Turnover Program Secured','LOW TV SECURED','SECURED - EMERGENCY CREDIT LINE','NORMAL INCOME PROGRAM SECURED','SECURE COVRES1','SMALL BUSINESS RES SEC','ABB PROGRAM BANK SECURED','SECURED - PRATHAM', 'MSME RES SEC', 'AUTO TOP UP PROG - SANJEEVANI SEC' ,'SECURED PEERLESS BUYOUT') THEN 'Sanjeevani_Secured'

   ELSE 'NA'

   END AS NEWPRODUCT,
  
   e.line_of_business,
   
   f.remarks as closure_remarks,
    
    g.status,
    g.First_disb_date,

   d.laf_number
 
from jayam_contract_details a

inner join jayam_contract_termination_dtl b on a.agreementid = b.agreementid


left join jayam_agreement_customer_info c on c.agreementid = a.agreementid and c.seq = 1

left join laf_detail_status d on d.laf_number = a.laf_id

left join tb_user_mst e on e.user_id = d.case_assigned_to_id

left join jayam_contract_termination_dtl f on f.agreementid = a.agreementid

left join (select distinct agreementid,status,min(jds.disb_date) as First_disb_date  from jayam_disbursement_schedule jds group by agreementid,status) g on g.agreementid = a.agreementid

where a.closuredate between '2026-08-01' and '2026-08-31'

"""

In [35]:
df9 = pd.read_sql_query(sql_query_5, connection)
# df9 = pd.read_sql_query(sql_query_5, conn)

In [36]:
df9.columns

Index(['agreementno', 'customername', 'productdescription', 'productflag',
       'schemedesc', 'prepay_penalty', 'branchdesc', 'effrate',
       'disbursaldate', 'future_principle', 'closuredate', 'day_diff',
       'newproduct', 'line_of_business', 'closure_remarks', 'status',
       'first_disb_date', 'laf_number'],
      dtype='object')

In [37]:
df9.shape

(1647, 18)

In [38]:
# condition = (df1['col1'] < 3) & (df['col2'] > 8)
# condition = (df9['closure_remarks'] == 'Disbursement Cancelled')

In [39]:
# df9 = df9[~condition] # ~ negates the condition

In [40]:
df9.columns

Index(['agreementno', 'customername', 'productdescription', 'productflag',
       'schemedesc', 'prepay_penalty', 'branchdesc', 'effrate',
       'disbursaldate', 'future_principle', 'closuredate', 'day_diff',
       'newproduct', 'line_of_business', 'closure_remarks', 'status',
       'first_disb_date', 'laf_number'],
      dtype='object')

In [41]:
df9.rename(columns={'agreementno':'lan'},inplace=True)

In [42]:
df10=pd.merge(df9,df8,on='lan',how='left')

In [43]:
print(df10[['lan', 'first_disb_date', 'disbursaldate']].head(10))

                    lan first_disb_date disbursaldate
0     UGHYDPS0000001763      2023-05-05    2023-05-05
1  HCFNAGSEC00001034831      2022-12-04    2022-12-04
2     UGMDPPS0000008809      2023-07-14    2023-07-14
3     UGPUNSU0000043611      2024-03-31    2024-03-31
4     UGDELSU0000064371      2024-09-30    2024-09-30
5     UGCMPPU0000053598      2024-07-22    2024-07-22
6     UGINDSS0000035408      2024-02-02    2024-02-02
7     UGSUPPU0000056481      2024-08-02    2024-08-02
8     UGSIKMS0000062253      2024-09-26    2024-09-26
9     UGDELMC0000076151      2025-01-31    2025-01-31


In [44]:
df10.columns

Index(['lan', 'customername', 'productdescription', 'productflag',
       'schemedesc', 'prepay_penalty', 'branchdesc', 'effrate',
       'disbursaldate', 'future_principle', 'closuredate', 'day_diff',
       'newproduct', 'line_of_business', 'closure_remarks', 'status',
       'first_disb_date', 'laf_number', 'object_pri_key_1', 'program_name',
       'purpose_of_loan', 'case_assigned_to_id', 'Solar_Ind'],
      dtype='object')

In [45]:
# df10.to_excel(r"C:\Users\saurabh.khopkar\Downloads\Data.xlsx", index=False)

In [46]:
sql_query_6 = """
select distinct lan ,lob_1,dsbd.c_business,dsbd.c_scheme,
dsbd.product_lob,
dsbd.partner_type
from disb_summary_base_data dsbd 
"""

In [47]:
df11 = pd.read_sql_query(sql_query_6, connection)

In [48]:
# df11 = pd.read_excel(r"D:\Saurabh\BIU\BIU Daily Disbursal.xlsx","disb_base_data")

In [49]:
# df11.rename(columns={'LAN':'lan'},inplace=True)

In [50]:
# neworder = ['lan','C_business', 'C_scheme', 'C_business1',
#        'line_of_business', 'Business_LOB', 'Product_LOB','New Format','Micro_br']

In [51]:
# df12=df11.reindex(columns=neworder)
df12=df11

In [52]:
df13=pd.merge(df10,df12,on='lan',how='left')

In [53]:
print(df13[['lan', 'first_disb_date', 'disbursaldate']].head(10))

                    lan first_disb_date disbursaldate
0     UGHYDPS0000001763      2023-05-05    2023-05-05
1  HCFNAGSEC00001034831      2022-12-04    2022-12-04
2     UGMDPPS0000008809      2023-07-14    2023-07-14
3     UGPUNSU0000043611      2024-03-31    2024-03-31
4     UGDELSU0000064371      2024-09-30    2024-09-30
5     UGCMPPU0000053598      2024-07-22    2024-07-22
6     UGINDSS0000035408      2024-02-02    2024-02-02
7     UGSUPPU0000056481      2024-08-02    2024-08-02
8     UGSIKMS0000062253      2024-09-26    2024-09-26
9     UGDELMC0000076151      2025-01-31    2025-01-31


In [54]:
df13['branchdesc'] = df13['branchdesc'].str.upper()

In [55]:
df13.columns


Index(['lan', 'customername', 'productdescription', 'productflag',
       'schemedesc', 'prepay_penalty', 'branchdesc', 'effrate',
       'disbursaldate', 'future_principle', 'closuredate', 'day_diff',
       'newproduct', 'line_of_business', 'closure_remarks', 'status',
       'first_disb_date', 'laf_number', 'object_pri_key_1', 'program_name',
       'purpose_of_loan', 'case_assigned_to_id', 'Solar_Ind', 'lob_1',
       'c_business', 'c_scheme', 'product_lob', 'partner_type'],
      dtype='object')

In [56]:
df14 = pd.read_excel(r"D:\Saurabh\S3 Bucket file\branch_code_state_mapping.xlsx")

In [57]:
df14['branch_name_Upper'] = df14['branch_name'].str.upper()

In [58]:
neworder = ['branch_name_Upper','branch_type', 'Micro_Phase']

In [59]:
df15=df14.reindex(columns=neworder)

In [60]:
df15.rename(columns={'branch_name_Upper':'branchdesc'},inplace=True)

In [61]:
df15

,branchdesc,branch_type,Micro_Phase
0,ADONI MICRO,Micro,3.0
1,ADONI MICRO,Micro,3.0
2,AGAR,Micro,3.0
3,AGRA MICRO,Micro,2.0
4,AGRA PRIME,Prime,NaN
...,...,...,...
520,YAMUNANAGAR,Micro,2.0
521,YAVATMAL MICRO,Micro,3.0
522,AHMEDABAD 1,Micro,4.0
523,MANSAROVAR JAIPUR EM2,PL,NaN


In [62]:
df16=pd.merge(df13,df15,on='branchdesc',how='left')

In [63]:
print(df16[['lan', 'first_disb_date', 'disbursaldate']].head(10))

                    lan first_disb_date disbursaldate
0     UGHYDPS0000001763      2023-05-05    2023-05-05
1  HCFNAGSEC00001034831      2022-12-04    2022-12-04
2     UGMDPPS0000008809      2023-07-14    2023-07-14
3     UGPUNSU0000043611      2024-03-31    2024-03-31
4     UGDELSU0000064371      2024-09-30    2024-09-30
5     UGCMPPU0000053598      2024-07-22    2024-07-22
6     UGINDSS0000035408      2024-02-02    2024-02-02
7     UGSUPPU0000056481      2024-08-02    2024-08-02
8     UGSIKMS0000062253      2024-09-26    2024-09-26
9     UGDELMC0000076151      2025-01-31    2025-01-31


In [64]:
# df16.to_excel(r"C:\Users\saurabh.khopkar\Downloads\Data.xlsx", index=False)
df16.columns

Index(['lan', 'customername', 'productdescription', 'productflag',
       'schemedesc', 'prepay_penalty', 'branchdesc', 'effrate',
       'disbursaldate', 'future_principle', 'closuredate', 'day_diff',
       'newproduct', 'line_of_business', 'closure_remarks', 'status',
       'first_disb_date', 'laf_number', 'object_pri_key_1', 'program_name',
       'purpose_of_loan', 'case_assigned_to_id', 'Solar_Ind', 'lob_1',
       'c_business', 'c_scheme', 'product_lob', 'partner_type', 'branch_type',
       'Micro_Phase'],
      dtype='object')

In [65]:
df16.loc[df16['branch_type'].isin(['Micro']),'Micro_br']=1

In [66]:
a = df16['lob_1'].unique()
a

array([nan, 'Gro Micro Sales Secured', 'Partnership and Alliances RE',
       'Branch Unsecured Sales', 'Sanjeevani Micro',
       'Branch Secured Sales', 'Machinery Ecosystem Business'],
      dtype=object)

In [67]:
df16['Foreclosure_LOB'] = np.where(df16['lob_1'].notnull(), df16['lob_1'],'NA')

In [68]:
# df16['Foreclosure_LOB'] = np.where(df16['New Format'] == 'Blank', df16['A'], df16['B'])
# df16['Foreclosure_LOB'] = np.where(df16['New Format'] != 'Blank', df16['New Format'],'0')
# df16['Foreclosure_LOB'] =  np.where(df16['Foreclosure_LOB'] == 'Blank' & df16['New Format'] != 'Blank', df16['New Format'],'')

In [69]:
df16['Source'] = np.where(df16['lob_1'].notnull(), 'DMS_Mapping','NA')

In [70]:
# df16 = df16.replace('NAN', np.nan)

In [71]:
df16.columns

Index(['lan', 'customername', 'productdescription', 'productflag',
       'schemedesc', 'prepay_penalty', 'branchdesc', 'effrate',
       'disbursaldate', 'future_principle', 'closuredate', 'day_diff',
       'newproduct', 'line_of_business', 'closure_remarks', 'status',
       'first_disb_date', 'laf_number', 'object_pri_key_1', 'program_name',
       'purpose_of_loan', 'case_assigned_to_id', 'Solar_Ind', 'lob_1',
       'c_business', 'c_scheme', 'product_lob', 'partner_type', 'branch_type',
       'Micro_Phase', 'Micro_br', 'Foreclosure_LOB', 'Source'],
      dtype='object')

In [72]:
a = df16['Foreclosure_LOB'].unique()
a

array(['NA', 'Gro Micro Sales Secured', 'Partnership and Alliances RE',
       'Branch Unsecured Sales', 'Sanjeevani Micro',
       'Branch Secured Sales', 'Machinery Ecosystem Business'],
      dtype=object)

In [73]:
# df16.to_excel(r"C:\Users\saurabh.khopkar\Downloads\df16.xlsx", index=False)

In [74]:
# df161 = df16(df16['lan'].isin(['UGSUMMS0000050698','HCFVJYSLC00001026716']) #Filtering dataframe
df161 = df16[(df16["lan"]=="UGSUMMS0000050698")] #Filtering dataframe
neworder = ['lan','lob_1','Foreclosure_LOB', 'Source',]
df162=df161.reindex(columns=neworder)
df162

,lan,lob_1,Foreclosure_LOB,Source


In [75]:
# conditions = [
#     (df16['Foreclosure_LOB'] == '') & (df16['Micro_br'] == 1) & 
#     (df16['schemedesc_x'] == ['GRO-MICRO-UNSECURED','SBL - UNSECURED','MICRO UNSECURED BLI'])]
# #     (df16['Set'] == 'Z') & (df16['Type'] == 'B'),
# #     (df16['Type'] == 'B')]
# # choices = ['yellow', 'blue', 'purple']
# df16['Foreclosure_LOB'] = np.select(conditions, default='Micro_Unsecured')

In [76]:
df16.loc[
#     df16['Foreclosure_LOB'].notnull() &
        df16['Foreclosure_LOB'].isin(['NA']) & 
        df16['Micro_br'].isin([1]) & 
        df16['schemedesc'].isin(['GRO-MICRO-UNSECURED','SBL - UNSECURED','MICRO UNSECURED BLI']),
        'Foreclosure_LOB']="Micro_Unsecured"

In [77]:

# # df16.loc[
# # df16.loc[df16['Foreclosure_LOB'].isnull() &
# df16.loc[
#     df16['Foreclosure_LOB']=='' &
#     df16['Micro_br']==1 & 
# # df16.loc[df16['Micro_br'].isin(['1']) & 
#     df16['schemedesc_x'].isin(['GRO-MICRO-UNSECURED','SBL - UNSECURED','MICRO UNSECURED BLI']),
#         'Foreclosure_LOB']="Micro_Unsecured"

In [78]:
# df16.to_excel(r"C:\Users\saurabh.khopkar\Downloads\df16.xlsx", index=False)

In [79]:
# df16['Source'].value_counts()
# df['color'] = np.where(df['Set']=='Z', 'green', 'red')

In [80]:
# # df16['Foreclosure_LOB'] = np.where((df16["Micro_br"]=='1') & (df16["Solar_Ind"]=='1'),"Micro_Solar","NA")
# df1['Mch_Intermediate'] = np.where(
# #     (df16['Foreclosure_LOB'].notnull()) &
#     (df1['lob'] == 'Plant and Machinery Intermediary Distribution'),
# #     (df16['Micro_br'] == 1) & 
# #     (df16['Solar_Ind'] == 1),
#     1,df1['Mch_Intermediate'])
df16['Foreclosure_LOB'] = np.where(
#     (df16['Foreclosure_LOB'].notnull()) &
    (df16['Micro_br'] == 'NA') & 
    (df16['Micro_br'] == 1) & 
    (df16['Solar_Ind'] == 1),
    "Micro_Solar",df16['Foreclosure_LOB'])

In [81]:
# df16.to_excel(r"C:\Users\saurabh.khopkar\Downloads\df16.xlsx", index=False)

In [82]:
# df16['Foreclosure_LOB'] = np.where(
#     (df16['Micro_br'] == '1') & 
#     (df16['schemedesc_x'].isin(['GRO-MICRO-UNSECURED','SBL - UNSECURED']),
#     "Micro_Unsecured",df16['Foreclosure_LOB'])

df16.loc[
#     df16['Foreclosure_LOB'].notnull() &
     df16['Foreclosure_LOB'].isin(['NA']) & 
        df16['Micro_br'].isin([1]) & 
# df16.loc[df16['Micro_br'].isin(['1']) & 
        df16['schemedesc'].isin(['GRO-MICRO-UNSECURED','SBL - UNSECURED','MICRO UNSECURED BLI']),
        'Foreclosure_LOB']="Micro_Unsecured"
    

In [83]:
# df16.to_excel(r"C:\Users\saurabh.khopkar\Downloads\df16.xlsx", index=False)

In [84]:
df16.loc[
#     df16['Foreclosure_LOB'].notnull() &
      df16['Foreclosure_LOB'].isin(['NA']) & 
        df16['Micro_br'].isin([1]) & 
# df16.loc[df16['Micro_br'].isin(['1']) & 
        df16['schemedesc'].isin(['AUTO TOP UP PROG - SANJEEVANI SEC','SAATHIPLUS - SECURED','SAATHIPRIME - SECURED','SECURED - SANJEEVANI','SAHYOG SECURED - AIP','SAHYOG SECURED - ABB']),
        'Foreclosure_LOB']="Micro Sanjeevani"

In [85]:
df16.loc[
#     df16['Foreclosure_LOB'].notnull() &
          df16['Foreclosure_LOB'].isin(['NA']) & 
        df16['Micro_br'].isin([1]) & 
# df16.loc[df16['Micro_br'].isin(['1']) & 
        df16['schemedesc'].isin(['LOW LTV MACHINERY LOAN','MTL - MACHINERY PROGRAM']),
        'Foreclosure_LOB']="Micro_Machinery"

In [86]:
df16.loc[
#     df16['Foreclosure_LOB'].notnull() & 
           df16['Foreclosure_LOB'].isin(['NA']) & 
        df16['Micro_br'].isin([1]) & 
# df16.loc[df16['Micro_br'].isin(['1']) & 
        df16['schemedesc'].isin(['GRO-MICRO-SECURED','MICRO SECURED BLI']),
        'Foreclosure_LOB']="Micro_Secured"

In [87]:
df16.loc[
#     df16['Foreclosure_LOB'].notnull() & 
           df16['Foreclosure_LOB'].isin(['NA']) & 
        df16['Micro_br'].isin([1]) & 
# df16.loc[df16['Micro_br'].isin(['1']) & 
        df16['schemedesc'].isin(['GRO-MICRO-SECURED','MICRO SECURED BLI']),
        'Foreclosure_LOB']="Micro_Secured"

In [88]:
a = df16['Foreclosure_LOB'].unique()
a

array(['NA', 'Micro_Secured', 'Gro Micro Sales Secured',
       'Partnership and Alliances RE', 'Micro Sanjeevani',
       'Micro_Unsecured', 'Branch Unsecured Sales', 'Sanjeevani Micro',
       'Branch Secured Sales', 'Micro_Machinery',
       'Machinery Ecosystem Business'], dtype=object)

In [89]:
df16.columns

Index(['lan', 'customername', 'productdescription', 'productflag',
       'schemedesc', 'prepay_penalty', 'branchdesc', 'effrate',
       'disbursaldate', 'future_principle', 'closuredate', 'day_diff',
       'newproduct', 'line_of_business', 'closure_remarks', 'status',
       'first_disb_date', 'laf_number', 'object_pri_key_1', 'program_name',
       'purpose_of_loan', 'case_assigned_to_id', 'Solar_Ind', 'lob_1',
       'c_business', 'c_scheme', 'product_lob', 'partner_type', 'branch_type',
       'Micro_Phase', 'Micro_br', 'Foreclosure_LOB', 'Source'],
      dtype='object')

In [90]:
df17 = pd.read_excel(r"D:\Saurabh\S3 Bucket file\machinery_users.xlsx")
# df17 = pd.read_excel(r"D:\Saurabh\BIU\BIU Daily Disbursal.xlsx","Ashish Shirke Team")

In [91]:
df17 = df17[(df17["Department"]=="Plant and Machinery Intermediary Distribution")] #Filtering dataframe

In [92]:
# df17.loc[
# #     df16['Foreclosure_LOB'].notnull() &
#           df17['Department'].isin(['Plant and Machinery Intermediary Distribution']),  
#         'Intermidiate']=1

In [93]:
df17= df17.drop(columns=['Department'])

In [94]:
df17

,Employee_Id


In [95]:
df17.rename(columns={'Employee_Id':'case_assigned_to_id'},inplace=True)
neworder = ['case_assigned_to_id','Intermidiate']
df18=df17.reindex(columns=neworder)

In [96]:
df19=pd.merge(df16,df18,on='case_assigned_to_id',how='left')

In [97]:
print(df19[['lan', 'first_disb_date', 'disbursaldate']].head(10))

                    lan first_disb_date disbursaldate
0     UGHYDPS0000001763      2023-05-05    2023-05-05
1  HCFNAGSEC00001034831      2022-12-04    2022-12-04
2     UGMDPPS0000008809      2023-07-14    2023-07-14
3     UGPUNSU0000043611      2024-03-31    2024-03-31
4     UGDELSU0000064371      2024-09-30    2024-09-30
5     UGCMPPU0000053598      2024-07-22    2024-07-22
6     UGINDSS0000035408      2024-02-02    2024-02-02
7     UGSUPPU0000056481      2024-08-02    2024-08-02
8     UGSIKMS0000062253      2024-09-26    2024-09-26
9     UGDELMC0000076151      2025-01-31    2025-01-31


In [98]:
a = df19['Source'].unique()
a

array(['NA', 'DMS_Mapping'], dtype=object)

In [99]:
df19.columns

Index(['lan', 'customername', 'productdescription', 'productflag',
       'schemedesc', 'prepay_penalty', 'branchdesc', 'effrate',
       'disbursaldate', 'future_principle', 'closuredate', 'day_diff',
       'newproduct', 'line_of_business', 'closure_remarks', 'status',
       'first_disb_date', 'laf_number', 'object_pri_key_1', 'program_name',
       'purpose_of_loan', 'case_assigned_to_id', 'Solar_Ind', 'lob_1',
       'c_business', 'c_scheme', 'product_lob', 'partner_type', 'branch_type',
       'Micro_Phase', 'Micro_br', 'Foreclosure_LOB', 'Source', 'Intermidiate'],
      dtype='object')

In [100]:
df19.loc[
    df19['Intermidiate']==1   ,
#     df19['Foreclosure_LOB'].notnull(),
        'Source']="Intermidiate_Manual_File"

In [101]:
df19.loc[
    df19['Intermidiate']==1   ,
#     df19['Foreclosure_LOB'].notnull(),
        'Foreclosure_LOB']="Plant and Machinery Intermediary Distribution"

In [102]:
df19['Source'].value_counts()

Source
NA             1509
DMS_Mapping     192
Name: count, dtype: int64

In [103]:
df19.loc[
#     df19['Foreclosure_LOB'].notnull() &
      df19['Foreclosure_LOB'].isin(['NA']) & 
        ~df19['Intermidiate'].isin([1]) & 
    df19['Solar_Ind'].isin([1]) & 
#     df19['Intermidiate']==1 & 
# df16.loc[df16['Micro_br'].isin(['1']) & 
        df19['line_of_business'].isin(['Branch Unsecured Sales','Small Business Loan','Branch Secured Sales']),
        'Foreclosure_LOB']="Intermidiate_Solar"

In [104]:
df19.columns

Index(['lan', 'customername', 'productdescription', 'productflag',
       'schemedesc', 'prepay_penalty', 'branchdesc', 'effrate',
       'disbursaldate', 'future_principle', 'closuredate', 'day_diff',
       'newproduct', 'line_of_business', 'closure_remarks', 'status',
       'first_disb_date', 'laf_number', 'object_pri_key_1', 'program_name',
       'purpose_of_loan', 'case_assigned_to_id', 'Solar_Ind', 'lob_1',
       'c_business', 'c_scheme', 'product_lob', 'partner_type', 'branch_type',
       'Micro_Phase', 'Micro_br', 'Foreclosure_LOB', 'Source', 'Intermidiate'],
      dtype='object')

In [105]:
# df19.loc[df19['Intermidiate']==1 & 
# # df16.loc[df16['Micro_br'].isin(['1']) & 
#         df19['line_of_business_x'].isin(['Branch Unsecured Sales','Small Business Loan']),
#         'Foreclosure_LOB']="Intermidiate_Solar"

In [106]:
df19.rename(columns={'schemedesc_x':'schemedesc'},inplace=True)

In [107]:
df20 = pd.read_excel(r"D:\Saurabh\S3 Bucket file\scheme to LOB.xlsx")

In [108]:
neworder = ['schemedesc','C_business', 'C_scheme']
# df2=df2.drop_duplicates()

In [109]:
df21=df20.reindex(columns=neworder)

In [110]:
df22=df21.drop_duplicates(subset=['schemedesc'])

In [111]:
df23=pd.merge(df19,df22,on='schemedesc',how='left')

In [112]:
print(df23[['lan', 'first_disb_date', 'disbursaldate']].head(10))

                    lan first_disb_date disbursaldate
0     UGHYDPS0000001763      2023-05-05    2023-05-05
1  HCFNAGSEC00001034831      2022-12-04    2022-12-04
2     UGMDPPS0000008809      2023-07-14    2023-07-14
3     UGPUNSU0000043611      2024-03-31    2024-03-31
4     UGDELSU0000064371      2024-09-30    2024-09-30
5     UGCMPPU0000053598      2024-07-22    2024-07-22
6     UGINDSS0000035408      2024-02-02    2024-02-02
7     UGSUPPU0000056481      2024-08-02    2024-08-02
8     UGSIKMS0000062253      2024-09-26    2024-09-26
9     UGDELMC0000076151      2025-01-31    2025-01-31


In [113]:
df23.columns

Index(['lan', 'customername', 'productdescription', 'productflag',
       'schemedesc', 'prepay_penalty', 'branchdesc', 'effrate',
       'disbursaldate', 'future_principle', 'closuredate', 'day_diff',
       'newproduct', 'line_of_business', 'closure_remarks', 'status',
       'first_disb_date', 'laf_number', 'object_pri_key_1', 'program_name',
       'purpose_of_loan', 'case_assigned_to_id', 'Solar_Ind', 'lob_1',
       'c_business', 'c_scheme', 'product_lob', 'partner_type', 'branch_type',
       'Micro_Phase', 'Micro_br', 'Foreclosure_LOB', 'Source', 'Intermidiate',
       'C_business', 'C_scheme'],
      dtype='object')

In [114]:
a = df23['Foreclosure_LOB'].unique()
a

array(['NA', 'Micro_Secured', 'Gro Micro Sales Secured',
       'Partnership and Alliances RE', 'Micro Sanjeevani',
       'Micro_Unsecured', 'Branch Unsecured Sales', 'Sanjeevani Micro',
       'Intermidiate_Solar', 'Branch Secured Sales', 'Micro_Machinery',
       'Machinery Ecosystem Business'], dtype=object)

In [115]:
# df23.loc[df23['branch_type']=='Prime' &
# # df23.loc[df23['productflag']=='SECLN' &
#          df23.loc[df23['productflag']=='SECLN',
# # df23.loc[df23['C_scheme_y']=='Sanjeevani',
# 'Foreclosure_LOB']=="Intermidiate_Secured"


df23.loc[
#     df23['Foreclosure_LOB'].notnull() &
    df23['Foreclosure_LOB'].isin(['NA']) & 
            ~df19['Intermidiate'].isin([1]) &     
    df23['branch_type'].isin(['Prime']) & 
        df23['productflag'].isin(['SECLN']) & 
        df23['C_scheme'].isin(['Sanjeevani']),
        'Foreclosure_LOB']="Intermidiate_Secured"

In [116]:

df23.loc[
#     df23['Foreclosure_LOB'].notnull() &
       df23['Foreclosure_LOB'].isin(['NA']) & 
            ~df19['Intermidiate'].isin([1]) &     
    df23['branch_type'].isin(['Prime']) & 
        df23['productflag'].isin(['USCLN']) & 
        df23['C_scheme'].isin(['Sanjeevani']),
        'Foreclosure_LOB']="Intermidiate_Unsecured"

In [117]:
df23.shape

(1701, 36)

In [118]:
df23['Foreclosure_LOB'].value_counts()

Foreclosure_LOB
NA                              878
Intermidiate_Unsecured          269
Micro_Secured                   262
Gro Micro Sales Secured          97
Partnership and Alliances RE     59
Intermidiate_Secured             41
Micro Sanjeevani                 37
Micro_Unsecured                  17
Branch Unsecured Sales           17
Sanjeevani Micro                  7
Branch Secured Sales              7
Machinery Ecosystem Business      5
Intermidiate_Solar                3
Micro_Machinery                   2
Name: count, dtype: int64

In [119]:
# list_values = ['KLUB WORKS UNSECURED']

In [120]:
df23.loc[
#     df23['Foreclosure_LOB'].notnull() &
       df23['Foreclosure_LOB'].isin(['NA']) & 
            ~df19['Intermidiate'].isin([1]) &     
    df23['branch_type'].isin(['Prime']) & 
#      df23['schemedesc'] not in (['KLUB WORKS UNSECURED']) &
#         df23['schemedesc']!=(['KLUB WORKS UNSECURED' , 'VELOCITY UNSECURED' , 'Velocity Unsecured New' ]) &
            ~df23['schemedesc'].isin(['KLUB WORKS UNSECURED' , 'VELOCITY UNSECURED' , 'Velocity Unsecured New']) &
#    list_values = ["KLUB WORKS UNSECURED"]
#     df23[~df23['schemedesc'].isin(list_values)]&
#     df23[~df23.schemedesc.isin('KLUB WORKS UNSECURED)] &
#     df23[~df23['schemedesc'].isin(list_values)]
#         df23['schemedesc']!= (['VELOCITY UNSECURED']) &
#         df23['schemedesc']!= (['Velocity Unsecured New']) &
#             df23['schemedesc']!= (['KLUB WORKS UNSECURED','VELOCITY UNSECURED','Velocity Unsecured New']) & 
#         df23['productflag']!=(['KLUB WORKS UNSECURED','VELOCITY UNSECURED']) & 
        df23['C_business'].isin(['Colending','Partnership']),
        'Foreclosure_LOB']="P&A_RE"

In [121]:
df23.loc[
#     df23['Foreclosure_LOB'].notnull() &
        df23['Foreclosure_LOB'].isin(['NA']) & 
            ~df19['Intermidiate'].isin([1]) &     
    df23['branch_type'].isin(['Prime']) & 
#      df23['schemedesc'] not in (['KLUB WORKS UNSECURED']) &
#         df23['schemedesc']!=(['KLUB WORKS UNSECURED' , 'VELOCITY UNSECURED' , 'Velocity Unsecured New' ]) &
            df23['schemedesc'].isin(['KLUB WORKS UNSECURED' , 'VELOCITY UNSECURED' , 'Velocity Unsecured New']) &
#    list_values = ["KLUB WORKS UNSECURED"]
#     df23[~df23['schemedesc'].isin(list_values)]&
#     df23[~df23.schemedesc.isin('KLUB WORKS UNSECURED)] &
#     df23[~df23['schemedesc'].isin(list_values)]
#         df23['schemedesc']!= (['VELOCITY UNSECURED']) &
#         df23['schemedesc']!= (['Velocity Unsecured New']) &
#             df23['schemedesc']!= (['KLUB WORKS UNSECURED','VELOCITY UNSECURED','Velocity Unsecured New']) & 
#         df23['productflag']!=(['KLUB WORKS UNSECURED','VELOCITY UNSECURED']) & 
        df23['C_business'].isin(['Colending','Partnership']),
        'Foreclosure_LOB']="P&A_DL"

In [122]:
df23.loc[
#     df23['Foreclosure_LOB'].notnull() &
        df23['Foreclosure_LOB'].isin(['NA']) & 
            ~df19['Intermidiate'].isin([1]) &     
    df23['branch_type'].isin(['Prime']) & 
#      df23['schemedesc'] not in (['KLUB WORKS UNSECURED']) &
#         df23['schemedesc']!=(['KLUB WORKS UNSECURED' , 'VELOCITY UNSECURED' , 'Velocity Unsecured New' ]) &
            df23['schemedesc'].isin(['SBL - UNSECURED']) &
#    list_values = ["KLUB WORKS UNSECURED"]
#     df23[~df23['schemedesc'].isin(list_values)]&
#     df23[~df23.schemedesc.isin('KLUB WORKS UNSECURED)] &
#     df23[~df23['schemedesc'].isin(list_values)]
#         df23['schemedesc']!= (['VELOCITY UNSECURED']) &
#         df23['schemedesc']!= (['Velocity Unsecured New']) &
#             df23['schemedesc']!= (['KLUB WORKS UNSECURED','VELOCITY UNSECURED','Velocity Unsecured New']) & 
#         df23['productflag']!=(['KLUB WORKS UNSECURED','VELOCITY UNSECURED']) & 
        ~df23['line_of_business'].isin(['Direct And Cross Sell']),
        'Foreclosure_LOB']="Intermidiate_SBL_Unsecured"

In [123]:
df23.loc[
#     df23['Foreclosure_LOB'].notnull() &
        df23['Foreclosure_LOB'].isin(['NA']) & 
            ~df19['Intermidiate'].isin([1]) &     
    df23['branch_type'].isin(['Prime']) & 
#      df23['schemedesc'] not in (['KLUB WORKS UNSECURED']) &
#         df23['schemedesc']!=(['KLUB WORKS UNSECURED' , 'VELOCITY UNSECURED' , 'Velocity Unsecured New' ]) &
#             df23['schemedesc'].isin(['SBL - UNSECURED']) &
#    list_values = ["KLUB WORKS UNSECURED"]
#     df23[~df23['schemedesc'].isin(list_values)]&
#     df23[~df23.schemedesc.isin('KLUB WORKS UNSECURED)] &
#     df23[~df23['schemedesc'].isin(list_values)]
#         df23['schemedesc']!= (['VELOCITY UNSECURED']) &
#         df23['schemedesc']!= (['Velocity Unsecured New']) &
#             df23['schemedesc']!= (['KLUB WORKS UNSECURED','VELOCITY UNSECURED','Velocity Unsecured New']) & 
#         df23['productflag']!=(['KLUB WORKS UNSECURED','VELOCITY UNSECURED']) & 
        df23['C_business'].isin(['Machinery']),
        'Foreclosure_LOB']="Ecosystem_Machinery"

In [124]:
df23.loc[
#     df23['Foreclosure_LOB'].notnull() &
        df23['Foreclosure_LOB'].isin(['NA']) & 
            ~df19['Intermidiate'].isin([1]) &     
    df23['branch_type'].isin(['Prime']) & 
#      df23['schemedesc'] not in (['KLUB WORKS UNSECURED']) &
#         df23['schemedesc']!=(['KLUB WORKS UNSECURED' , 'VELOCITY UNSECURED' , 'Velocity Unsecured New' ]) &
#             df23['schemedesc'].isin(['SBL - UNSECURED']) &
#    list_values = ["KLUB WORKS UNSECURED"]
#     df23[~df23['schemedesc'].isin(list_values)]&
#     df23[~df23.schemedesc.isin('KLUB WORKS UNSECURED)] &
#     df23[~df23['schemedesc'].isin(list_values)]
#         df23['schemedesc']!= (['VELOCITY UNSECURED']) &
#         df23['schemedesc']!= (['Velocity Unsecured New']) &
#             df23['schemedesc']!= (['KLUB WORKS UNSECURED','VELOCITY UNSECURED','Velocity Unsecured New']) & 
#         df23['productflag']!=(['KLUB WORKS UNSECURED','VELOCITY UNSECURED']) & 
#         df23['C_business_y'].isin(['Direct And Cross Sell','Cross Sell']),
    df23['line_of_business'].isin(['Direct And Cross Sell','Cross Sell','Cross-sell']),
        'Foreclosure_LOB']="Cross-Sell"

In [125]:
df23.loc[
#     df23['Foreclosure_LOB'].notnull() &
        df23['Foreclosure_LOB'].isin(['NA']) & 
            ~df19['Intermidiate'].isin([1]) &     
    df23['branch_type'].isin(['Prime']) & 
#      df23['schemedesc'] not in (['KLUB WORKS UNSECURED']) &
#         df23['schemedesc']!=(['KLUB WORKS UNSECURED' , 'VELOCITY UNSECURED' , 'Velocity Unsecured New' ]) &
            df23['C_scheme'].isin(['Saathi','Secured','Pratham']) &
#    list_values = ["KLUB WORKS UNSECURED"]
#     df23[~df23['schemedesc'].isin(list_values)]&
#     df23[~df23.schemedesc.isin('KLUB WORKS UNSECURED)] &
#     df23[~df23['schemedesc'].isin(list_values)]
#         df23['schemedesc']!= (['VELOCITY UNSECURED']) &
#         df23['schemedesc']!= (['Velocity Unsecured New']) &
#             df23['schemedesc']!= (['KLUB WORKS UNSECURED','VELOCITY UNSECURED','Velocity Unsecured New']) & 
#         df23['productflag']!=(['KLUB WORKS UNSECURED','VELOCITY UNSECURED']) & 
        df23['C_business'].isin(['Branch']),
        'Foreclosure_LOB']="Intermidiate_Secured"

In [126]:
df23.loc[
#     df23['Foreclosure_LOB'].notnull() &
        df23['Foreclosure_LOB'].isin(['NA']) & 
            ~df19['Intermidiate'].isin([1]) &     
    df23['branch_type'].isin(['Prime']) & 
#      df23['schemedesc'] not in (['KLUB WORKS UNSECURED']) &
#         df23['schemedesc']!=(['KLUB WORKS UNSECURED' , 'VELOCITY UNSECURED' , 'Velocity Unsecured New' ]) &
            df23['schemedesc'].isin(['SECURED - PRATHAM','SECURED - SANJEEVANI']) &
#    list_values = ["KLUB WORKS UNSECURED"]
#     df23[~df23['schemedesc'].isin(list_values)]&
#     df23[~df23.schemedesc.isin('KLUB WORKS UNSECURED)] &
#     df23[~df23['schemedesc'].isin(list_values)]
#         df23['schemedesc']!= (['VELOCITY UNSECURED']) &
#         df23['schemedesc']!= (['Velocity Unsecured New']) &
#             df23['schemedesc']!= (['KLUB WORKS UNSECURED','VELOCITY UNSECURED','Velocity Unsecured New']) & 
#         df23['productflag']!=(['KLUB WORKS UNSECURED','VELOCITY UNSECURED']) & 
        df23['C_business'].isin(['Branch']),
        'Foreclosure_LOB']="Intermidiate_Secured"

In [127]:
df23.loc[
#     df23['Foreclosure_LOB'].notnull() &
        df23['Foreclosure_LOB'].isin(['NA']) & 
            ~df19['Intermidiate'].isin([1]) &     
    df23['branch_type'].isin(['Prime']) & 
#      df23['schemedesc'] not in (['KLUB WORKS UNSECURED']) &
#         df23['schemedesc']!=(['KLUB WORKS UNSECURED' , 'VELOCITY UNSECURED' , 'Velocity Unsecured New' ]) &
            df23['C_scheme'].isin(['Unsecured']) &
#    list_values = ["KLUB WORKS UNSECURED"]
#     df23[~df23['schemedesc'].isin(list_values)]&
#     df23[~df23.schemedesc.isin('KLUB WORKS UNSECURED)] &
#     df23[~df23['schemedesc'].isin(list_values)]
#         df23['schemedesc']!= (['VELOCITY UNSECURED']) &
#         df23['schemedesc']!= (['Velocity Unsecured New']) &
#             df23['schemedesc']!= (['KLUB WORKS UNSECURED','VELOCITY UNSECURED','Velocity Unsecured New']) & 
#         df23['productflag']!=(['KLUB WORKS UNSECURED','VELOCITY UNSECURED']) & 
        df23['C_business'].isin(['Branch']),
        'Foreclosure_LOB']="Intermidiate_Unsecured"

In [128]:
df23.loc[
#     df23['Foreclosure_LOB'].notnull() &
        df23['Foreclosure_LOB'].isin(['NA']) & 
            ~df19['Intermidiate'].isin([1]) &     
    df23['branch_type'].isin(['Prime']) & 
#      df23['schemedesc'] not in (['KLUB WORKS UNSECURED']) &
#         df23['schemedesc']!=(['KLUB WORKS UNSECURED' , 'VELOCITY UNSECURED' , 'Velocity Unsecured New' ]) &
#             df23['C_scheme_y'].isin(['Unsecured']) &
#     df23['productdescription'].isin(['SANJEEVANI UN Secured']) &
#    list_values = ["KLUB WORKS UNSECURED"]
#     df23[~df23['schemedesc'].isin(list_values)]&
#     df23[~df23.schemedesc.isin('KLUB WORKS UNSECURED)] &
#     df23[~df23['schemedesc'].isin(list_values)]
#         df23['schemedesc']!= (['VELOCITY UNSECURED']) &
#         df23['schemedesc']!= (['Velocity Unsecured New']) &
#             df23['schemedesc']!= (['KLUB WORKS UNSECURED','VELOCITY UNSECURED','Velocity Unsecured New']) & 
#         df23['productflag']!=(['KLUB WORKS UNSECURED','VELOCITY UNSECURED']) & 
        df23['productdescription'].isin(['SANJEEVANI UN Secured']),
        'Foreclosure_LOB']="Intermidiate_Unsecured"

In [129]:
df23.loc[
#     df23['Foreclosure_LOB'].notnull() &
#        df23['Foreclosure_LOB'].isin(['NA']) & 
        df23['branch_type'].isin(['Prime']) & 
        
    df23['C_business'].isin(['Branch']) & 
#      df23['schemedesc'] not in (['KLUB WORKS UNSECURED']) &
#         df23['schemedesc']!=(['KLUB WORKS UNSECURED' , 'VELOCITY UNSECURED' , 'Velocity Unsecured New' ]) &
            ~df23['Intermidiate'].isin([1]) &
#    list_values = ["KLUB WORKS UNSECURED"]
#     df23[~df23['schemedesc'].isin(list_values)]&
#     df23[~df23.schemedesc.isin('KLUB WORKS UNSECURED)] &
#     df23[~df23['schemedesc'].isin(list_values)]
#         df23['schemedesc']!= (['VELOCITY UNSECURED']) &
#         df23['schemedesc']!= (['Velocity Unsecured New']) &
#             df23['schemedesc']!= (['KLUB WORKS UNSECURED','VELOCITY UNSECURED','Velocity Unsecured New']) & 
#         df23['productflag']!=(['KLUB WORKS UNSECURED','VELOCITY UNSECURED']) & 
        df23['Solar_Ind'].isin([1]),
        'Foreclosure_LOB']="Intermidiate_Solar"

In [130]:
df23.loc[
#     df23['Foreclosure_LOB'].notnull() &
#        df23['Foreclosure_LOB'].isin(['NA']) & 
        df23['branch_type'].isin(['Prime']) & 
        
    df23['C_business'].isin(['Machinery']) & 
#      df23['schemedesc'] not in (['KLUB WORKS UNSECURED']) &
#         df23['schemedesc']!=(['KLUB WORKS UNSECURED' , 'VELOCITY UNSECURED' , 'Velocity Unsecured New' ]) &
            ~df23['Intermidiate'].isin([1]) &
#    list_values = ["KLUB WORKS UNSECURED"]
#     df23[~df23['schemedesc'].isin(list_values)]&
#     df23[~df23.schemedesc.isin('KLUB WORKS UNSECURED)] &
#     df23[~df23['schemedesc'].isin(list_values)]
#         df23['schemedesc']!= (['VELOCITY UNSECURED']) &
#         df23['schemedesc']!= (['Velocity Unsecured New']) &
#             df23['schemedesc']!= (['KLUB WORKS UNSECURED','VELOCITY UNSECURED','Velocity Unsecured New']) & 
#         df23['productflag']!=(['KLUB WORKS UNSECURED','VELOCITY UNSECURED']) & 
        df23['Solar_Ind'].isin([1]),
        'Foreclosure_LOB']="Ecosystem_Solar"

In [131]:
df23.columns

Index(['lan', 'customername', 'productdescription', 'productflag',
       'schemedesc', 'prepay_penalty', 'branchdesc', 'effrate',
       'disbursaldate', 'future_principle', 'closuredate', 'day_diff',
       'newproduct', 'line_of_business', 'closure_remarks', 'status',
       'first_disb_date', 'laf_number', 'object_pri_key_1', 'program_name',
       'purpose_of_loan', 'case_assigned_to_id', 'Solar_Ind', 'lob_1',
       'c_business', 'c_scheme', 'product_lob', 'partner_type', 'branch_type',
       'Micro_Phase', 'Micro_br', 'Foreclosure_LOB', 'Source', 'Intermidiate',
       'C_business', 'C_scheme'],
      dtype='object')

In [132]:
df23.loc[
#     df23['Foreclosure_LOB'].notnull() &
       df23['Foreclosure_LOB'].isin(['NA']) & 
#         
#     df23['branch_type'].isin(['Prime']) & 
#      df23['schemedesc'] not in (['KLUB WORKS UNSECURED']) &
#         df23['schemedesc']!=(['KLUB WORKS UNSECURED' , 'VELOCITY UNSECURED' , 'Velocity Unsecured New' ]) &
            ~df23['schemedesc'].isin(['KLUB WORKS UNSECURED' , 'VELOCITY UNSECURED' , 'Velocity Unsecured New']) &
#    list_values = ["KLUB WORKS UNSECURED"]
#     df23[~df23['schemedesc'].isin(list_values)]&
#     df23[~df23.schemedesc.isin('KLUB WORKS UNSECURED)] &
#     df23[~df23['schemedesc'].isin(list_values)]
#         df23['schemedesc']!= (['VELOCITY UNSECURED']) &
#         df23['schemedesc']!= (['Velocity Unsecured New']) &
#             df23['schemedesc']!= (['KLUB WORKS UNSECURED','VELOCITY UNSECURED','Velocity Unsecured New']) & 
#         df23['productflag']!=(['KLUB WORKS UNSECURED','VELOCITY UNSECURED']) & 
        df23['C_business'].isin(['Colending','Partnership']),
        'Foreclosure_LOB']="P&A_RE"

In [133]:

df23.loc[
#     df23['Foreclosure_LOB'].notnull() &
    df23['Foreclosure_LOB'].isin(['NA']) & 
            ~df19['Intermidiate'].isin([1]) &     
    df23['branch_type'].isin(['Prime']) & 
        df23['productflag'].isin(['SECLN']),
#         df23['C_scheme_y'].isin(['Sanjeevani']),
        'Foreclosure_LOB']="Intermidiate_Secured"

In [134]:

df23.loc[
#     df23['Foreclosure_LOB'].notnull() &
    df23['Foreclosure_LOB'].isin(['NA']) & 
            ~df19['Intermidiate'].isin([1]) &     
    df23['branch_type'].isin(['Prime']) & 
        df23['schemedesc'].isin(['ABB PROGRAM BANK UNSECURED']) & 
        df23['productdescription'].isin(['P&A UNSECURED']),
#         df23['C_scheme_y'].isin(['Sanjeevani']),
        'Foreclosure_LOB']="P&A_RE"

In [135]:

df23.loc[
#     df23['Foreclosure_LOB'].notnull() &
    df23['Foreclosure_LOB'].isin(['NA']) & 
            df19['branchdesc'].isin(['WARDHA MICRO']) &     
    df23['productdescription'].isin(['GRO-MICRO-SECURED']) , 
#         df23['schemedesc'].isin(['ABB PROGRAM BANK UNSECURED']) & 
#         df23['productdescription'].isin(['MACHINERY']),
#         df23['C_scheme_y'].isin(['Sanjeevani']),
        'Foreclosure_LOB']="Micro_Machinery"

In [136]:
df161 = df23[(df23["lan"]=="UGWHAMS0000078875")] #Filtering dataframe
neworder = ['lan','lob_1','Foreclosure_LOB', 'Source',]
df162=df161.reindex(columns=neworder)
df162

,lan,lob_1,Foreclosure_LOB,Source


In [137]:

df23.loc[
#     df23['Foreclosure_LOB'].notnull() &
    df23['Foreclosure_LOB'].isin(['NA']) & 
            ~df23['Intermidiate'].isin([1]) &     
    df23['branch_type'].isin(['Micro']) & 
#         df23['schemedesc'].isin(['ABB PROGRAM BANK UNSECURED']) & 
        df23['productdescription'].isin(['MACHINERY']),
#         df23['C_scheme_y'].isin(['Sanjeevani']),
        'Foreclosure_LOB']="Micro_Machinery"

In [138]:

df23.loc[
#     df23['Foreclosure_LOB'].notnull() &
    df23['Foreclosure_LOB'].isin(['NA']) & 
            df23['schemedesc'].isin(['PURPLE FINANCE','SME RECUR']) ,     
#     df23['branch_type'].isin(['Micro']) & 
# #         df23['schemedesc'].isin(['ABB PROGRAM BANK UNSECURED']) & 
#         df23['productdescription'].isin(['MACHINERY']),
#         df23['C_scheme_y'].isin(['Sanjeevani']),
        'Foreclosure_LOB']="P&A_RE"

In [139]:
df23.loc[
        df23['Foreclosure_LOB'].isin(['Prime Secured','Branch Secured Sales']),
        'Foreclosure_LOB']="Intermidiate_Secured"

In [140]:
df23.loc[
        df23['Foreclosure_LOB'].isin(['Micro Sanjeevani']),
        'Foreclosure_LOB']="Micro_Sanjeevani"

In [141]:
df23.loc[
        df23['Foreclosure_LOB'].isin(['Prime UnSecured','Branch Unsecured Sales']),
        'Foreclosure_LOB']="Intermidiate_Unsecured"

In [142]:
df23.loc[
        df23['Foreclosure_LOB'].isin(['Prime machinery','Machinery Ecosystem Business']),
        'Foreclosure_LOB']="Ecosystem_Machinery"

In [143]:
df23.loc[
        df23['Foreclosure_LOB'].isin(['Prime SBL Unsecured']),
        'Foreclosure_LOB']="Intermidiate_SBL_Unsecured"

In [144]:
df23.loc[
        df23['Foreclosure_LOB'].isin(['Micro Secured','Gro Micro Sales Secured']),
        'Foreclosure_LOB']="Micro_Secured"

In [145]:
df23.loc[
        df23['Foreclosure_LOB'].isin(['Gro Micro Sales Unsecured']),
        'Foreclosure_LOB']="Micro_Unsecured"

In [146]:
df23.loc[
        df23['Foreclosure_LOB'].isin(['Sanjeevani Micro']),
        'Foreclosure_LOB']="Micro_Sanjeevani"

In [147]:
df23.loc[
        df23['Foreclosure_LOB'].isin(['Micro machinery','Machinery Micro']),
        'Foreclosure_LOB']="Micro_Machinery"

In [148]:
df23.loc[
        df23['Foreclosure_LOB'].isin(['Partnership/Allience_RE','Partnership and Alliances RE']),
        'Foreclosure_LOB']="P&A_RE"

In [149]:
df23.loc[
        df23['Foreclosure_LOB'].isin(['Partnership and Alliances DL']),
        'Foreclosure_LOB']="P&A_DL"

In [150]:
df23.loc[
        df23['Foreclosure_LOB'].isin(['Direct And Cross Sell']),
        'Foreclosure_LOB']="Cross-Sell"

In [151]:
df23.loc[
#     df16['Foreclosure_LOB'].notnull() & 
           df23['Foreclosure_LOB'].isin(['NA']) & 
        df23['C_business'].isin(['Secured','Unsecured']) , 
# df16.loc[df16['Micro_br'].isin(['1']) & 
        # df16['schemedesc'].isin(['GRO-MICRO-SECURED','MICRO SECURED BLI']),
        'Foreclosure_LOB']="Micro_Secured"

In [152]:
df23.loc[
#     df16['Foreclosure_LOB'].notnull() & 
           df23['Foreclosure_LOB'].isin(['NA']) & 
        df23['newproduct'].isin(['Partnership_Colending']) , 
# df16.loc[df16['Micro_br'].isin(['1']) & 
        # df16['schemedesc'].isin(['GRO-MICRO-SECURED','MICRO SECURED BLI']),
        'Foreclosure_LOB']="P&A_RE"

In [153]:
df23.loc[
#     df16['Foreclosure_LOB'].notnull() & 
           df23['Foreclosure_LOB'].isin(['Cross-Sell']) & 
        df23['branch_type'].isin(['Prime']) &
df23['newproduct'].isin(['Sanjeevani_Secured','Saathi_Secured']) ,
# df16.loc[df16['Micro_br'].isin(['1']) & 
        # df16['schemedesc'].isin(['GRO-MICRO-SECURED','MICRO SECURED BLI']),
        'Foreclosure_LOB']="Branch_Secured"

In [154]:
df23.loc[
#     df16['Foreclosure_LOB'].notnull() & 
           df23['Foreclosure_LOB'].isin(['Cross-Sell']) & 
        df23['branch_type'].isin(['Prime']) &
df23['newproduct'].isin(['Sanjeevani_UNSecured']) ,
# df16.loc[df16['Micro_br'].isin(['1']) & 
        # df16['schemedesc'].isin(['GRO-MICRO-SECURED','MICRO SECURED BLI']),
        'Foreclosure_LOB']="Branch_Unsecured"

In [155]:
df23.loc[
#     df16['Foreclosure_LOB'].notnull() & 
           df23['Foreclosure_LOB'].isin(['Cross-Sell']) & 
        df23['branch_type'].isin(['Prime']) &
df23['productdescription'].isin(['SANJEEVANI UN Secured']) ,
# df16.loc[df16['Micro_br'].isin(['1']) & 
        # df16['schemedesc'].isin(['GRO-MICRO-SECURED','MICRO SECURED BLI']),
        'Foreclosure_LOB']="Branch_Unsecured"

In [156]:
df23.loc[
#     df16['Foreclosure_LOB'].notnull() & 
           df23['Foreclosure_LOB'].isin(['Cross-Sell']) & 
        df23['branch_type'].isin(['Micro']) &
df23['productdescription'].isin(['SANJEEVANI Secured']) ,
# df16.loc[df16['Micro_br'].isin(['1']) & 
        # df16['schemedesc'].isin(['GRO-MICRO-SECURED','MICRO SECURED BLI']),
        'Foreclosure_LOB']="Micro_Secured"

In [157]:
df23.loc[
#     df16['Foreclosure_LOB'].notnull() & 
           df23['Foreclosure_LOB'].isin(['NA']) & 
        # df23['branch_type'].isin(['Micro']) &
df23['productdescription'].isin(['GRO-MICRO-SECURED','GRO-MICRO-UNSECURED']),
# df16.loc[df16['Micro_br'].isin(['1']) & 
        # df16['schemedesc'].isin(['GRO-MICRO-SECURED','MICRO SECURED BLI']),
        'Foreclosure_LOB']="Micro_Secured"

In [158]:
df23.loc[
#     df16['Foreclosure_LOB'].notnull() & 
           df23['Foreclosure_LOB'].isin(['Intermidiate_SBL_Unsecured','Intermidiate_Unsecured']) , 
#         df16['Micro_br'].isin([1]) & 
# # df16.loc[df16['Micro_br'].isin(['1']) & 
#         df16['schemedesc'].isin(['GRO-MICRO-SECURED','MICRO SECURED BLI']),
        'Foreclosure_LOB']="Branch_Unsecured"

In [159]:
df23.loc[
#     df16['Foreclosure_LOB'].notnull() & 
           df23['Foreclosure_LOB'].isin(['Intermidiate_Secured']) , 
#         df16['Micro_br'].isin([1]) & 
# # df16.loc[df16['Micro_br'].isin(['1']) & 
#         df16['schemedesc'].isin(['GRO-MICRO-SECURED','MICRO SECURED BLI']),
        'Foreclosure_LOB']="Branch_Secured"

In [160]:
df23.loc[
#     df16['Foreclosure_LOB'].notnull() & 
           df23['Foreclosure_LOB'].isin(['Intermidiate_Solar','Plant and Machinery Intermediary Distribution']) , 
#         df16['Micro_br'].isin([1]) & 
# # df16.loc[df16['Micro_br'].isin(['1']) & 
#         df16['schemedesc'].isin(['GRO-MICRO-SECURED','MICRO SECURED BLI']),
        'Foreclosure_LOB']="Intermidiate_Equipment_finance"

In [161]:
df23.loc[
#     df16['Foreclosure_LOB'].notnull() & 
           df23['Foreclosure_LOB'].isin(['Micro_Secured','Micro_Unsecured']) , 
#         df16['Micro_br'].isin([1]) & 
# # df16.loc[df16['Micro_br'].isin(['1']) & 
#         df16['schemedesc'].isin(['GRO-MICRO-SECURED','MICRO SECURED BLI']),
        'Foreclosure_LOB']="Micro_Secured"

In [162]:
df23.loc[
    df23['Foreclosure_LOB'].isin(['NA']) & 
           df23['schemedesc'].isin(['AUTO TOP UP PROG - SANJEEVANI UNSEC']), 
           # df23['schemedesc'].isin(['AUTO TOP UP PROG - SANJEEVANI UNSEC']) & 
#         df16['Micro_br'].isin([1]) & 
# # df16.loc[df16['Micro_br'].isin(['1']) & 
#         df16['schemedesc'].isin(['GRO-MICRO-SECURED','MICRO SECURED BLI']),
        'Foreclosure_LOB']="Branch_Unsecured"

In [163]:
df23['Closure_Amt_Cr'] = df23['future_principle']/10000000

In [164]:
# df161 = df23[(df23["lan"]=="UGDELMC0000056356")] #Filtering dataframe
# neworder = ['lan','New Format','Foreclosure_LOB', 'Source',]
# df162=df161.reindex(columns=neworder)
# df162

In [165]:
df24 = df23.sort_values("lan").drop_duplicates(["lan", "closuredate","future_principle"], keep="first")

In [166]:
print(df24[['lan', 'first_disb_date', 'disbursaldate']].head(10))

                       lan first_disb_date disbursaldate
1497      110001UKN0034111      2020-11-24    2020-11-24
1557      400001USL0027156      2020-08-12    2020-08-12
338       500001SEC0017932      2019-12-31    2019-12-31
1284      500001USL0020580      2020-02-13    2020-02-13
1637      600001SEC0033898      2020-11-21    2020-11-21
488     AVUSFDELSB00010053      2019-07-08    2019-07-08
1673  HCFAHMSEC00001002997      2021-05-07    2021-05-07
1171  HCFAHMSEC00001008017      2021-09-30    2021-09-30
559   HCFAHMSEC00001034404      2022-12-03    2022-12-03
416   HCFAHMSEC00001042151      2023-02-25    2023-02-25


In [167]:
df161 = df24[(df24["lan"]=="HCFMAHUSC00001033516")] #Filtering dataframe
neworder = ['lan','closuredate','future_principle']
df162=df161.reindex(columns=neworder)
df162

,lan,closuredate,future_principle


In [168]:
# df23.to_excel(r"C:\Users\saurabh.khopkar\Downloads\foreclosure_ason_Jan'25_26.xlsx", index=False)
# df = df.drop(columns=['column_nameA', 'column_nameB'])

In [169]:
df24['closuredate'] = pd.to_datetime(df24['closuredate'])
df24['closuredate'] = pd.to_datetime(df24['closuredate'])
df24['closure_Month'] = (df24['closuredate'] + pd.offsets.MonthEnd(0)).dt.date
df24['closure_Month'] = pd.to_datetime(df24['closure_Month'])

In [170]:
df24['disbursaldate'] = pd.to_datetime(df24['disbursaldate'])
df24['disbursal_Month'] = (df24['disbursaldate'] + pd.offsets.MonthEnd(0)).dt.date
df24['disbursal_Month'] = pd.to_datetime(df24['disbursal_Month'])

In [171]:
df24.loc[
#     ~df24['Foreclosure_LOB'].notnull() &
        ~df24['Foreclosure_LOB'].isin([]) &
        df24['status'].isin(['Disbursement Cancelled']), 
#         df16['Micro_br'].isin([1]) & 
#         df16['schemedesc'].isin(['GRO-MICRO-UNSECURED','SBL - UNSECURED','MICRO UNSECURED BLI']),
#         'closure_remarks']="Micro_Unsecured"
    'closure_remarks']=df24['status']

In [172]:
df24['Remarks_Status'] = np.where(df24['Foreclosure_LOB'].notnull(), 0,0)

In [173]:
df24.columns

Index(['lan', 'customername', 'productdescription', 'productflag',
       'schemedesc', 'prepay_penalty', 'branchdesc', 'effrate',
       'disbursaldate', 'future_principle', 'closuredate', 'day_diff',
       'newproduct', 'line_of_business', 'closure_remarks', 'status',
       'first_disb_date', 'laf_number', 'object_pri_key_1', 'program_name',
       'purpose_of_loan', 'case_assigned_to_id', 'Solar_Ind', 'lob_1',
       'c_business', 'c_scheme', 'product_lob', 'partner_type', 'branch_type',
       'Micro_Phase', 'Micro_br', 'Foreclosure_LOB', 'Source', 'Intermidiate',
       'C_business', 'C_scheme', 'Closure_Amt_Cr', 'closure_Month',
       'disbursal_Month', 'Remarks_Status'],
      dtype='object')

In [174]:
# print(df24[['are_equal']])

In [175]:
# df24['disbursal_Month_num'] = pd.to_datetime(df24['disbursaldate_Month']).dt.month
# df24['closuredate_Month_num'] = pd.to_datetime(df24['closuredate_Month']).dt.month

In [176]:
 df24['are_equal'] = df24['closure_Month']==df24['disbursal_Month']

In [177]:
df24.loc[
        ~df24['closure_remarks'].isin(['Disbursement Cancelled','OTC Loan Cancellation']) &
        ~df24['are_equal'].isin(['TRUE']),
        'Remarks_Status']=1

In [178]:
# condition = (df1['col1'] < 3) & (df['col2'] > 8)
condition = (df24['closure_remarks'] == 'Disbursement Cancelled')

In [179]:
# df24 = df24[~condition] # ~ negates the condition

In [180]:
df161 = df24[(df24["lan"]=="UGAAIMS0000083352")] #Filtering dataframe
neworder = ['lan','closure_remarks']
df162=df161.reindex(columns=neworder)
df162

,lan,closure_remarks


In [181]:
df24=df24

In [182]:
neworder = ['Foreclosure_LOB','Closure_Amt_Cr','closuredate','branch_type','Micro_Phase','Micro_br',
            'lan','customername','productdescription','productflag','schemedesc','prepay_penalty','branchdesc',
            'effrate','disbursaldate','first_disb_date','future_principle','day_diff','newproduct','line_of_business','status',
            'laf_number','object_pri_key_1','program_name','purpose_of_loan','case_assigned_to_id',
            'Solar_Ind','lob_1','c_business','c_scheme','product_lob','partner_type','Source',
            'Intermidiate','C_business','C_scheme','closure_Month','disbursal_Month','closure_remarks','Remarks_Status','are_equal']

In [183]:
df24A=df24.reindex(columns=neworder)

In [184]:
print(df24A[['lan', 'first_disb_date', 'disbursaldate']].head(10))

                       lan first_disb_date disbursaldate
1497      110001UKN0034111      2020-11-24    2020-11-24
1557      400001USL0027156      2020-08-12    2020-08-12
338       500001SEC0017932      2019-12-31    2019-12-31
1284      500001USL0020580      2020-02-13    2020-02-13
1637      600001SEC0033898      2020-11-21    2020-11-21
488     AVUSFDELSB00010053      2019-07-08    2019-07-08
1673  HCFAHMSEC00001002997      2021-05-07    2021-05-07
1171  HCFAHMSEC00001008017      2021-09-30    2021-09-30
559   HCFAHMSEC00001034404      2022-12-03    2022-12-03
416   HCFAHMSEC00001042151      2023-02-25    2023-02-25


In [185]:
# df24 = pd.DataFrame(df24, index=['Intermidiate_Secured','Intermidiate_Unsecured',
#                                  'Intermidiate_SBL_Unsecured','Intermidiate_Solar',
#                                  'Machinery Intermidiate','Micro_Secured',
#                                  'Micro_Unsecured','Micro_Sanjeevani','Micro Solar',
#                                  'Ecosystem_Machinery','P&A_DL','P&A_RE','Partnership/Allience','NA',])

In [186]:
# Foreclosure_LOB = ["Intermidiate_Secured","Intermidiate_Unsecured",
#                                  "Intermidiate_SBL_Unsecured","Intermidiate_Solar",
#                                  "Machinery Intermidiate","Micro_Secured",
#                                  "Micro_Unsecured","Micro_Sanjeevani","Micro Solar",
#                                  "Ecosystem_Machinery","P&A_DL","P&A_RE","Partnership/Allience","NA"]

In [187]:
# cats = ['Intermidiate_Unsecured',
#                                  'Intermidiate_SBL_Unsecured','Intermidiate_Solar',
#                                  'Machinery Intermidiate','Micro_Secured',
#                                  'Micro_Unsecured','Micro_Sanjeevani','Micro Solar',
#                                  'Ecosystem_Machinery','P&A_DL','P&A_RE','Partnership/Allience','NA']
# cat_type = CategoricalDtype(categories=cats, ordered=True)
# df24['Foreclosure_LOB'] = df24['Foreclosure_LOB'].astype(cat_type)

In [188]:
#  df24["Foreclosure_LOB"] = df24["Foreclosure_LOB"].astype('category', categories=["Intermidiate_Secured","Intermidiate_Unsecured",
#                                  "Intermidiate_SBL_Unsecured","Intermidiate_Solar",
#                                  "Machinery Intermidiate","Micro_Secured",
#                                  "Micro_Unsecured","Micro_Sanjeevani","Micro Solar",
#                                  "Ecosystem_Machinery","P&A_DL","P&A_RE","Partnership/Allience","NA"])

In [189]:
# df24.pivot(index="lan",columns="Foreclosure_LOB",values="future_principle")
# df24.pivot(index="laf_number",columns="Foreclosure_LOB",values="future_principle")
# df25 = df24.pivot_table(values='future_principle', index=['Foreclosure_LOB'],
#                        columns=['C'], aggfunc="sum", fill_value=0)
df25 = df24A.pivot_table(values='Closure_Amt_Cr', index=['Foreclosure_LOB'],
                        aggfunc="sum", fill_value=0, margins=True,margins_name= 'TOTAL')

In [190]:
# df26 = pd.DataFrame(df25, index=['Intermidiate_Secured','Intermidiate_Unsecured',
#                                  'Intermidiate_SBL_Unsecured','Intermidiate_Solar',
#                                  'Machinery Intermidiate','Micro_Secured',
#                                  'Micro_Unsecured','Micro_Sanjeevani','Micro Solar',
#                                  'Ecosystem_Machinery','P&A_DL','P&A_RE','Partnership/Allience','NA'])

In [191]:
df25

,Closure_Amt_Cr
Foreclosure_LOB,
Branch_Secured,69.771498
Branch_Unsecured,20.138681
Ecosystem_Machinery,17.842976
Intermidiate_Equipment_finance,0.055146
Micro_Machinery,0.410282
Micro_Sanjeevani,25.546094
Micro_Secured,43.537087
NA,0.349437
P&A_DL,0.000000


In [192]:
# df26 = df25.reindex_axis(['Intermidiate_Secured','Intermidiate_Unsecured','Intermidiate_SBL_Unsecured','Intermidiate_Solar','Machinery Intermidiate','Micro_Secured','Micro_Unsecured','Micro_Sanjeevani','Micro Solar','Ecosystem_Machinery','P&A_DL','P&A_RE','Partnership/Allience','NA'], axis=1)

In [193]:
# df=pd.pivot_table(index=['ID','CATEG','LEVEL'],columns='COLS',values=['VALUE'])

In [194]:
# writer = pd.ExcelWriter(r"C:\Users\saurabh.khopkar\Downloads\df25.xlsx", engine='xlsxwriter')
# df25.to_excel(writer,sheet_name='dataframe')
# writer.save()

In [195]:
file_path = 'C:\\Users\\saurabh.khopkar\\Downloads\\'
one_day_ago = datetime.now() - timedelta(days=1)
# cur_time = one_day_ago.strftime("%d-%m-%Y")
cur_time = one_day_ago.strftime("%d-%m-%Y_%H-%M-%S")
filename="foreclosure_ason_" + str(cur_time) + ".xlsx"
# df.to_excel(file_path + filename, index=False)

In [196]:
writer = pd.ExcelWriter(file_path + filename, engine = 'xlsxwriter')

In [197]:
df24A.to_excel(writer, sheet_name = "foreclosue_base", index = False)

In [198]:
for column in df24A:
    column_length = max(df24A[column].astype(str).map(len).max(), len(column))
    col_idx = df24A.columns.get_loc(column)
    writer.sheets['foreclosue_base'].set_column(col_idx, col_idx, column_length)

In [199]:
df25.to_excel(writer, sheet_name = "foreclosue_pivot")

In [200]:
for column in df25:
    column_length = max(df25[column].astype(str).map(len).max(), len(column))
    col_idx = df25.columns.get_loc(column)
    writer.sheets['foreclosue_pivot'].set_column(col_idx, col_idx, column_length)

In [201]:
# df24A.to_excel(writer, sheet_name = "foreclosue_base", index = False)

In [202]:
# df25.to_excel(writer, sheet_name = "foreclosue_pivot")

In [203]:
writer.close()